# Build & Chat with a Workflow

This notebook shows how to build a runnable workflow in one shot and then hold a
live, multi-turn conversation with it:

1. Assemble a fully-hydrated workflow config (nodes, edges, and LLM configs)
2. Set a workflow-level default LLM plus a per-node LLM override
3. Route between nodes with a conditional edge
4. Create the workflow (a `v0` version is published and activated automatically)
5. Drive a multi-turn interactive chat through an `AsyncWorkflowHandle`

> **Tip:** Node and edge configs are built with the typed classes from
> `interactly.configs` (requires `pip install "interactly[configs]"`).
> A plain-dict form is also supported — see `06_interactly_configs.ipynb`.


> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. Create a workflow from config

We assemble a two-node support workflow with `WorkflowConfigFullyHydrated` and create it
in one call with `create_from_config()`. The config demonstrates layered LLM configuration
and conditional routing:

- **Greeting node** (start node) has no `llms_config`, so it falls back to the **workflow-level
  default** — Google `gemini-3.5-flash`.
- **Farewell node** overrides the default with a **node-level** OpenAI `gpt-5.4-mini` config,
  and does not wait for user input (`wait_for_user_message=False`) so it ends the chat.
- A **conditional edge** routes from greeting to farewell only when the user wants to end the chat.

As soon as the workflow is created, a `v0` version is published and marked active.


In [ ]:
from interactly.configs import (
    ConditionalEdgeConfig,
    ConditionConfig,
    SayLLMNodeConfig,
    PromptConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
    OpenAILLMConfig,
    OPENAIModel,
    GoogleLLMConfig,
    GOOGLEModel,
)
from interactly.types.workflows.workflow import Workflow

# greeting_node sets no llms_config, so it inherits the workflow-level default LLM (Gemini, below)
greeting_node = SayLLMNodeConfig(
    name="Greet User",
    is_start=True,
    main_response_config=PromptConfig(
        prompt="You are a friendly support agent. Greet the user warmly and ask how you can help.",
    )
)

farewell_node = SayLLMNodeConfig(
    name="Farewell User",
    main_response_config=PromptConfig(
        prompt="You are a friendly support agent. Say goodbye to the user warmly and thank them for their time.",
    ),
    # Node-level LLM config overrides the workflow default: OpenAI gpt-5.4-mini
    llms_config=OpenAILLMConfig(
        model=OPENAIModel.GPT_5_4_MINI,
        max_tokens=256,
    ),
    self_loop=False,
    wait_for_user_message=False
)

config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(
        name="Customer Support Agent",
        description="Simple two-node support workflow from 02_interactive_workflow.ipynb",
        # Workflow-level default LLM config (used by nodes without their own llms_config): Google gemini-3.5-flash
        llms_config=GoogleLLMConfig(
            model=GOOGLEModel.GEMINI_3_5_FLASH,
            max_tokens=512,
            thinking_budget=128,
        ),
    ),
    node_configs=[greeting_node, farewell_node],
    edge_configs=[
        # Conditional edge: route greeting -> farewell ONLY when the user EXPLICITLY ends the chat.
        # This condition is evaluated by the LLM on every turn, so keep it strict. A loose condition
        # (e.g. "the user wants to end the chat") can fire early — on a greeting or a support question —
        # ending the run before later turns execute and leaving fewer input/output pairs than turns sent.
        ConditionalEdgeConfig(
            source_node_logical_id=greeting_node.logical_id,
            destination_node_logical_id=farewell_node.logical_id,
            condition=ConditionConfig(
                condition_freeform=(
                    "Take this path ONLY if, in their most recent message, the user explicitly asks to "
                    "end, stop, or finish the conversation, or says goodbye. Do NOT take this path merely "
                    "because the user greets you, asks a question, or mentions having a support ticket or "
                    "an issue — in those cases stay in the conversation."
                )
            ),
        )
    ],
)

workflow: Workflow = await client.workflows.create_from_config(config)

print(f"Created workflow  id={workflow.id}  name={workflow.name!r}")

WF_ID = workflow.id
FAREWELL_NODE_LOGICAL_ID = farewell_node.logical_id
GREETING_NODE_LOGICAL_ID = greeting_node.logical_id

## 2. Execute the workflow — interactive chat

Because `v0` is already active, we can chat with the workflow right away.

We drive a **multi-turn conversation** with an `AsyncWorkflowHandle`, which tracks
the `run_id` across turns for us. Each turn we send a user message wrapped in a
`WorkflowRunInput` (thread `"0"` is the main conversation thread) and print the
assistant replies as chat bubbles. The first turn uses `WorkflowCommand.START`;
subsequent turns use `WorkflowCommand.DATA`. After each turn we also dump the raw
typed events so you can inspect exactly what the runtime emitted.


In [ ]:
from langchain_core.messages import HumanMessage

from interactly.runtime.handle import AsyncWorkflowHandle
from interactly.runtime.events import (
    AssistantResponseEvent,
    BusyWaitForUserMessageEvent,
)
from interactly.configs import (
    LLMNodeRunInput,
    NodesRunInputs,
    WorkflowCommand,
    WorkflowRunInput,
)

# A handle wraps WF_ID and tracks the run_id across turns for us.
chat: AsyncWorkflowHandle = await client.workflows.handle(WF_ID)


async def send_message(user_text: str, *, command: WorkflowCommand = WorkflowCommand.DATA):
    """Send one user turn, print the assistant's reply bubbles, and return the raw events.

    The user's message is delivered on thread "0" (the main conversation thread).
    """
    print(f"User: {user_text}")

    run_input = WorkflowRunInput(
        command=command,
        thread_to_node_inputs={
            "0": NodesRunInputs(
                node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content=user_text)])]
            )
        },
    )

    events = []
    async for event in chat.arun(run_input):
        events.append(event)
        if isinstance(event, AssistantResponseEvent) and event.content:
            print(f"  Assistant: {event.content}")
        elif isinstance(event, BusyWaitForUserMessageEvent):
            print(f"  (waiting for user at {event.origin_node_name})")
    return events

In [ ]:
# Turn 1 kicks off the session with START; later turns use DATA (the default).
events = await send_message("Hello", command=WorkflowCommand.START)

print(f"\n\n----Events from Turn 1----")
for event in events:
    print(f"type={type(event).__name__}: {event.model_dump_json(indent=2)}")


In [ ]:
events = await send_message("hi. have a support ticket")

print(f"\n\n----Events from Turn 2----")
for event in events:
    print(f"type={type(event).__name__}: {event.model_dump_json(indent=2)}")


In [ ]:
events = await send_message("actually i want to end the conversation")

print(f"\n\n----Events from Turn 3----")

for event in events:
    print(f"type={type(event).__name__}: {event.model_dump_json(indent=2)}")


print(f"\nSession run_id: {chat.run_id}")
RUN_ID = chat.run_id

## 3. Fetch the completed run

In [ ]:
from interactly import Run

run: Run = await client.runs.get(RUN_ID)
print(f"Run {run.id}  status={run.status}  started={run.started_at}")

In [ ]:
# Print every input/output pair of the completed run, plus the events each turn generated.
#
# `run.input_output_pairs` is a list of `WorkflowRunInputOutputPair` (from
# interactly_configs; falls back to plain dicts if the [configs] extra is absent).
# Each pair corresponds to a contiguous execution segment of the workflow:
#   - pair.run_input               -> the WorkflowRunInput that drove the segment
#   - pair.run_output.events       -> the events emitted (assistant responses,
#                                     user messages, node/edge events, ...)
from interactly.runtime.events import parse_event


def _get(obj, key):
    """Read `key` from a pydantic model or a plain dict (graceful w/o [configs])."""
    return getattr(obj, key, None) if not isinstance(obj, dict) else obj.get(key)


for i, pair in enumerate(run.input_output_pairs):
    print(f"\n\n================= Pair {i} ========================\n\n")

    run_input = _get(pair, "run_input")
    print(f"  Input: {run_input!r}")

    run_output = _get(pair, "run_output")
    events = _get(run_output, "events") or []
    print(f"  Output: {len(events)} event(s)")
    for raw in events:
        try:
            event = parse_event(raw)
            print(f"    type={type(event).__name__}: {event.model_dump_json(indent=2)}")
        except Exception:
            print(f"    (raw) {raw}")

## 5. Cleanup

In [ ]:
await client.workflows.delete(WF_ID)
print(f"Workflow {WF_ID} deleted.")

await client.close()